# 07 · Statistical robustness of the scale selection

Quantifies the uncertainty and stability of the multi-criterion scale ranking, tests it against
the shared-temperature and land-cover concerns, and evaluates the categorical detection skill and
the yield models. This notebook reflects the revised analysis: the fifth selection criterion is the
**real-time ratio r₀/r_max** (independent of the Pearson correlation), not the earlier zero-lag
correlation.

**Sections**
1. Composite score (five criteria; real-time ratio as the independent fifth)
2. Bootstrap (year-block + district-cluster, 1000 replicates each)
3. Cross-validation (leave-one-year-out, leave-one-district-out)
4. Land-cover mask sensitivity (six rainfed districts, İpsala excluded)
5. Shared-temperature analysis (VCI-only vs TCI-only vs VHI)
6. Figure 9 (bootstrap robustness)
7. Detection skill — full contingency table + block-bootstrap CIs
8. Yield-model comparison — AIC, marginal R², leave-one-year-out RMSE

**Headline:** SPEI-3 is selected first in 69 % (year-block) and 83 % (district-cluster) of
replicates and in 40/40 and 7/7 cross-validation folds; excluding İpsala raises the VHI–SPEI-3
correlation from 0.471 to 0.521; the 3-month optimum holds against VCI-only, TCI-only and VHI; the
detection HSS is 0.28 (positive in 100 % of replicates); and the yield-optimal scale is longer
(SPEI-9/12) than the vegetation-condition optimum, as expected.

Inputs: `../data/derived/`; the yield section also needs the TÜİK file (see `../data/README.md`).

## 0 · Setup

In [ ]:
import pandas as pd, numpy as np, unicodedata, warnings
warnings.filterwarnings('ignore')
from scipy import stats
from sklearn.feature_selection import mutual_info_regression
import matplotlib.pyplot as plt, matplotlib as mpl
mpl.rcParams.update({'font.size':9,'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':150,'savefig.dpi':300})
DERIVED='../data/derived/'; FIGDIR='../figures/'
TR=str.maketrans('ıİşŞçÇğĞöÖüÜ','iissccggoouu')
def fold(s):
    s=str(s).translate(TR); return ''.join(c for c in unicodedata.normalize('NFKD',s) if not unicodedata.combining(c)).lower().strip()
vhi=pd.read_csv(DERIVED+'trakya_district_vhi_vci_tci_monthly.csv'); vhi['VHI']=0.5*vhi['VCI']+0.5*vhi['TCI']; vhi['key']=vhi['name'].map(fold)
S=pd.read_csv(DERIVED+'trakya_spi_spei_1965_2024.csv'); S['key']=S['name'].map(fold)
m=vhi.merge(S.drop(columns='name'),on=['key','year','month'],how='inner')
SCALES=[1,3,6,9,12]; IDX=[f'{f}{k}' for f in ['SPI','SPEI'] for k in SCALES]; WIN=[6,7,8,9,10]
d=m[m.month.isin(WIN)].dropna(subset=['VHI']).copy()
print(f'{len(d)} district-months | {d.name.nunique()} districts | {d.year.nunique()} years')

## 1 · Composite score with the independent fifth criterion

The fifth criterion is the **real-time ratio** r₀/r_max: the concurrent (zero-lag) VHI correlation
divided by the maximum correlation over lags −3…+3 months. It measures how much of an index's
achievable association is realised concurrently and, unlike the earlier zero-lag correlation, is not
a monotone function of the Pearson magnitude — so it is genuinely independent of the first criterion.
Mutual information uses the k-NN estimator (k = 5, fixed seed).

In [ ]:
def realtime_ratio(idx, frame=m):
    """r0 / r_max over lags -3..+3, computed on the full monthly frame, scored on Jun-Oct."""
    prof={}; t=frame.sort_values(['key','year','month']).copy()
    for L in range(-3,4):
        t['lg']=t.groupby('key')[idx].shift(L)
        s=t[t.month.isin(WIN)]; x=s['VHI'].values; y=s['lg'].values; ok=np.isfinite(x)&np.isfinite(y)
        prof[L]=stats.pearsonr(x[ok],y[ok])[0] if ok.sum()>50 else np.nan
    r0=prof[0]; rmax=np.nanmax(list(prof.values()))
    return r0/rmax if rmax>0 else np.nan

RT={idx:realtime_ratio(idx) for idx in IDX}

def composite(df, seed=0, rt=RT):
    rows=[]
    for idx in IDX:
        x=df['VHI'].values; y=df[idx].values; ok=np.isfinite(x)&np.isfinite(y)
        if ok.sum()<30: return None
        xv,yv=x[ok],y[ok]
        pe=stats.pearsonr(xv,yv)[0]; sp=stats.spearmanr(xv,yv)[0]
        mi=mutual_info_regression(yv.reshape(-1,1),xv,random_state=seed,n_neighbors=5)[0]
        q1,q3=np.quantile(yv,[1/3,2/3]); do=xv[yv>=q3].mean()-xv[yv<=q1].mean()
        rows.append({'idx':idx,'pearson':pe,'spearman':sp,'MI':mi,'dose':do,'rt':rt[idx]})
    c=pd.DataFrame(rows)
    for cc in ['pearson','spearman','MI','dose','rt']: c[cc+'_r']=c[cc].rank()
    c['composite']=c[[cc+'_r' for cc in ['pearson','spearman','MI','dose','rt']]].sum(axis=1)
    return c

base=composite(d).sort_values('composite',ascending=False)
print(base[['idx','pearson','spearman','MI','dose','rt','composite']].round(3).to_string(index=False))
print(f"\nTop: {base.iloc[0]['idx']} (composite {base.iloc[0]['composite']:.1f}/50)")
print("SPEI-3 rank on each criterion:",
      {cc:int(base[cc].rank(ascending=False)[base.idx=='SPEI3'].iloc[0]) for cc in ['pearson','spearman','MI','dose','rt']})

## 2 · Bootstrap (year-block + district-cluster)

The real-time ratio is a structural property of each index's lag profile, so it is computed once on
the full record and held fixed across resamples; the other four criteria are recomputed on every
replicate. Runtime: a few minutes.

In [ ]:
RTrank=pd.Series(RT).rank()
def composite_fast(win):
    rows=[]
    for idx in IDX:
        x=win['VHI'].values; y=win[idx].values; ok=np.isfinite(x)&np.isfinite(y)
        if ok.sum()<30: return None
        xv,yv=x[ok],y[ok]
        rows.append({'idx':idx,'pearson':stats.pearsonr(xv,yv)[0],'spearman':stats.spearmanr(xv,yv)[0],
                     'MI':mutual_info_regression(yv.reshape(-1,1),xv,random_state=0,n_neighbors=5)[0],
                     'dose':xv[yv>=np.quantile(yv,2/3)].mean()-xv[yv<=np.quantile(yv,1/3)].mean()})
    c=pd.DataFrame(rows).set_index('idx')
    comp=c['pearson'].rank()+c['spearman'].rank()+c['MI'].rank()+c['dose'].rank()+RTrank.reindex(c.index)
    return comp.idxmax(), c.loc['SPEI3','pearson']

B=1000; years=sorted(d.year.unique()); dists=sorted(d.name.unique())
rngY=np.random.default_rng(42); rngD=np.random.default_rng(7)
byY={y:d[d.year==y] for y in years}; byD={x:d[d.name==x] for x in dists}
winY={}; winD={}; pear3=[]
for b in range(B):
    r=composite_fast(pd.concat([byY[y] for y in rngY.choice(years,len(years),replace=True)],ignore_index=True))
    if r: winY[r[0]]=winY.get(r[0],0)+1; pear3.append(r[1])
    r2=composite_fast(pd.concat([byD[x] for x in rngD.choice(dists,len(dists),replace=True)],ignore_index=True))
    if r2: winD[r2[0]]=winD.get(r2[0],0)+1
nY=sum(winY.values()); nD=sum(winD.values())
print("Year-block:",{k:f'{100*v/nY:.1f}%' for k,v in sorted(winY.items(),key=lambda kv:-kv[1]) if v>3})
print("District-cluster:",{k:f'{100*v/nD:.1f}%' for k,v in sorted(winD.items(),key=lambda kv:-kv[1]) if v>3})
p3=np.array(pear3); print(f"SPEI-3 r 95% CI: [{np.percentile(p3,2.5):.3f}, {np.percentile(p3,97.5):.3f}]")

## 3 · Cross-validation

In [ ]:
loyo={}; lodo={}
for y in years:
    r=composite_fast(d[d.year!=y]); loyo[r[0]]=loyo.get(r[0],0)+1
for x in dists:
    r=composite_fast(d[d.name!=x]); lodo[r[0]]=lodo.get(r[0],0)+1
print("Leave-one-year-out:",loyo,f"of {len(years)}")
print("Leave-one-district-out:",lodo,f"of {len(dists)}")

## 4 · Land-cover mask sensitivity — six rainfed districts

Excluding the irrigated İpsala valley (where the static CORINE-211 mask does not fully isolate
rainfed cropland) tests robustness to land-cover misclassification.

In [ ]:
rf=d[d.name!='Ipsala']
c=composite(rf).sort_values('composite',ascending=False)
print(c[['idx','pearson','spearman','composite']].round(3).head(4).to_string(index=False))
print(f"\n-> top {c.iloc[0]['idx']}, VHI–SPEI-3 r = "
      f"{c.loc[c.idx=='SPEI3','pearson'].iloc[0]:.3f} (vs 0.471 with all 7 districts)")

## 5 · Shared-temperature analysis

SPEI (PET) and VHI (TCI) both contain temperature, so part of SPEI's advantage over SPI could be a
shared thermal signal. Ranking the families against VCI-only (α = 1, no temperature), TCI-only
(α = 0), and the VHI (α = 0.5) separates the effects.

In [ ]:
def corr(a,b):
    ok=np.isfinite(a)&np.isfinite(b); return stats.pearsonr(a[ok],b[ok])[0]

for alpha,lab in [(1.0,'VCI-only (no temperature)'),(0.0,'TCI-only (thermal)'),(0.5,'VHI (alpha=0.5)')]:
    v=vhi.copy(); v['REF']=alpha*v['VCI']+(1-alpha)*v['TCI']
    dd=v.merge(S.drop(columns='name'),on=['key','year','month']).query('month in @WIN').dropna(subset=['REF'])
    rs={}
    for f in ['SPI','SPEI']:
        rs[f]={k:corr(dd['REF'].values, dd[f+str(k)].values) for k in SCALES}
    bspi=max(rs['SPI'],key=rs['SPI'].get); bspei=max(rs['SPEI'],key=rs['SPEI'].get)
    print(f"{lab:28} best SPI: SPI-{bspi} ({rs['SPI'][bspi]:.3f}) | "
          f"best SPEI: SPEI-{bspei} ({rs['SPEI'][bspei]:.3f}) | "
          f"SPEI-3 {rs['SPEI'][3]:.3f} vs SPI-3 {rs['SPI'][3]:.3f}")
print("\nThe 3-month scale is optimal against every reference; SPI-3 beats SPEI-3 against VCI-only,")
print("confirming that SPEI's VHI-margin is partly the temperature shared by PET and TCI.")

## 6 · Figure 9 — bootstrap robustness

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(9.6,3.8),gridspec_kw={'width_ratios':[1.15,1]})
order=['SPEI3','SPI3','SPEI6','SPI9','SPEI1','SPEI9','SPI6','SPI1','SPI12','SPEI12']
lab=[o.replace('SPEI','SPEI-').replace('SPI','SPI-') for o in order]
fy=[100*winY.get(o,0)/nY for o in order]; fd=[100*winD.get(o,0)/nD for o in order]
yy=np.arange(len(order)); h=0.38
ax[0].barh(yy-h/2,fy,h,color='#c0392b',label=f'Year-block (B={nY})')
ax[0].barh(yy+h/2,fd,h,color='#2980b9',label=f'District-cluster (B={nD})')
ax[0].set_yticks(yy); ax[0].set_yticklabels(lab,fontsize=8); ax[0].invert_yaxis()
ax[0].set_xlabel('Frequency selected as top-ranked index (%)'); ax[0].set_xlim(0,100)
ax[0].set_title('(a) Bootstrap selection frequency',loc='left',fontweight='bold',fontsize=10); ax[0].legend(fontsize=7.5,loc='lower right')
for i,(a_,b_) in enumerate(zip(fy,fd)):
    if a_>2: ax[0].text(a_+1,yy[i]-h/2,f'{a_:.0f}%',va='center',fontsize=7,color='#c0392b')
    if b_>2: ax[0].text(b_+1,yy[i]+h/2,f'{b_:.0f}%',va='center',fontsize=7,color='#2980b9')
ax[1].hist(p3,bins=35,color='#c0392b',alpha=0.7,density=True)
ax[1].axvline(np.percentile(p3,2.5),color='#800',ls=':'); ax[1].axvline(np.percentile(p3,97.5),color='#800',ls=':')
ax[1].set_xlabel('Bootstrap VHI–SPEI-3 Pearson r'); ax[1].set_ylabel('density')
ax[1].set_title('(b) Correlation uncertainty',loc='left',fontweight='bold',fontsize=10)
fig.tight_layout(); fig.savefig(FIGDIR+'Fig_bootstrap.png',bbox_inches='tight'); plt.show()

## 7 · Detection skill — contingency table and block-bootstrap CIs

Frequency-matched threshold: a month is flagged when VHI falls below the value giving the same
drought frequency as SPEI-3 < −1.

In [ ]:
dd=m[m.month.isin(WIN)].dropna(subset=['VHI','SPEI3']).copy()
def cont(ref,det):
    a=int((ref&det).sum()); b=int((~ref&det).sum()); c=int((ref&~det).sum()); e=int((~ref&~det).sum()); n=a+b+c+e
    POD=a/(a+c); FAR=b/(a+b); CSI=a/(a+b+c); bias=(a+b)/(a+c)
    exp=((a+b)*(a+c)+(c+e)*(b+e))/n; HSS=(a+e-exp)/(n-exp)
    return dict(a=a,b=b,c=c,d=e,n=n,POD=POD,FAR=FAR,CSI=CSI,bias=bias,HSS=HSS)
ref=dd['SPEI3']<-1.0; vt=np.quantile(dd['VHI'].values,ref.mean()); M=cont(ref,dd['VHI']<vt)
print(f"Contingency (SPEI-3<-1 vs VHI<{vt:.1f}, n={M['n']}):")
print(f"  hits {M['a']} | misses {M['c']} | false alarms {M['b']} | correct-neg {M['d']}")
yrs=sorted(dd.year.unique()); rng=np.random.default_rng(11); byy={y:dd[dd.year==y] for y in yrs}
boot={k:[] for k in ['POD','FAR','CSI','HSS']}
for b in range(2000):
    bd=pd.concat([byy[y] for y in rng.choice(yrs,len(yrs),replace=True)],ignore_index=True)
    r=bd['SPEI3']<-1.0
    if r.sum()<5: continue
    mm=cont(r, bd['VHI']<np.quantile(bd['VHI'].values,r.mean()))
    for k in boot: boot[k].append(mm[k])
for k in ['POD','FAR','CSI','HSS']:
    a=np.array(boot[k]); print(f"  {k}: {M[k]:.3f}  95% CI [{np.percentile(a,2.5):.3f}, {np.percentile(a,97.5):.3f}]")
print(f"  P(HSS>0) = {100*(np.array(boot['HSS'])>0).mean():.1f}%")

## 8 · Yield-model comparison (requires the TÜİK yield file)

Compares SPEI scales as predictors of sunflower yield by AIC, marginal R² and leave-one-year-out
RMSE. Place `Yield.xls` beside this notebook to run.

In [ ]:
import os
YIELD='Yield.xls'
if not os.path.exists(YIELD):
    print('Yield file not found — skipping (see ../data/README.md).')
else:
    import statsmodels.formula.api as smf
    KEEP7=['corlu','edirne','ipsala','kirklareli','luleburgaz','tekirdag','uzunkopru']
    y=pd.read_excel(YIELD,sheet_name='Sunflower',header=0)
    yl=y.melt(id_vars='Year',var_name='name',value_name='yield').rename(columns={'Year':'year'}); yl['key']=yl['name'].map(fold)
    yl=yl[yl.key.isin(KEEP7)]
    sp=S[S.month==8][['key','year']+[f'SPEI{k}' for k in SCALES]]
    df=yl.merge(sp,on=['key','year']).dropna(subset=['yield']); df['year_c']=df.year-df.year.mean()
    yrs=sorted(df.year.unique())
    print(f"Sunflower (7 districts, n={len(df)}):")
    print(f"{'scale':8}{'coef':>8}{'AIC':>9}{'margR2':>9}{'LOYO_RMSE':>11}")
    for k in SCALES:
        s=df.dropna(subset=[f'SPEI{k}']).copy()
        md_=smf.mixedlm(f"Q('yield') ~ SPEI{k} + year_c", s, groups=s['key']).fit(reml=False)
        fe=np.var(md_.fittedvalues); r2=fe/(fe+float(md_.cov_re.iloc[0,0])+md_.scale)
        errs=[]
        for yy in yrs:
            tr=s[s.year!=yy]; te=s[s.year==yy]
            if len(te)==0 or len(tr)<20: continue
            mt=smf.mixedlm(f"Q('yield') ~ SPEI{k} + year_c", tr, groups=tr['key']).fit(reml=False)
            pr=mt.params['Intercept']+mt.params[f'SPEI{k}']*te[f'SPEI{k}']+mt.params['year_c']*te['year_c']
            errs+=list(te['yield'].values-pr.values)
        print(f"SPEI-{k:<3}{md_.params[f'SPEI{k}']:>8.1f}{md_.aic:>9.1f}{r2:>9.3f}{np.sqrt(np.mean(np.array(errs)**2)):>11.1f}")
    print("\nYield-optimal scale (AIC/R2/RMSE) is longer than the vegetation-condition optimum (SPEI-3):")
    print("concurrent VHI tracks current stress (short scale); cumulative yield integrates the season (long scale).")

---
### Summary
| Test | Result |
|---|---|
| Year-block bootstrap | SPEI-3 first in ~69 % |
| District-cluster bootstrap | SPEI-3 first in ~83 % |
| Leave-one-year-out / district-out | 40/40 · 7/7 |
| VHI–SPEI-3 r, 95 % CI | [0.386, 0.545] |
| Rainfed-only (İpsala excluded) | SPEI-3, r rises to 0.521 |
| Shared-temperature | 3-month optimal vs every reference; SPI-3 > SPEI-3 vs VCI-only |
| Detection HSS | 0.28 [0.16, 0.37], P(HSS>0)=100 % |
| Yield-optimal scale | SPEI-9/12 (longer than vegetation optimum) |

The fifth criterion (real-time ratio) is independent of the Pearson correlation, so the composite is
not double-weighted; SPEI-3 remains the selected scale for concurrent vegetation-condition monitoring.
